# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
print("Goodbye World")

Goodbye World


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [10]:
from __future__ import annotations

import os
from typing import Literal

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI
from pydantic import BaseModel, Field
from pathlib import Path


In [25]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')


from langchain_community.document_loaders import PyPDFLoader

#file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
file_path = Path("pdf/Managing Oneself_Drucker_HBR.pdf")
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
13


In [ ]:

api_gateway_key = os.getenv("API_GATEWAY_KEY")
if not api_gateway_key:
    raise RuntimeError("API_GATEWAY_KEY missing in ../05_src/.secrets")


client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

context_text = "\n".join(page.page_content for page in docs)

# Join pages into one context string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


ToneType = Literal["Formal Academic Writing"]

class ArticleBrief(BaseModel):
    Author: str = Field(...)
    Title: str = Field(...)
    Relevance: str = Field(..., description="Single paragraph.")
    Summary: str = Field(..., description="<= 1000 tokens.")
    Tone: ToneType = Field(...)
    InputTokens: int = Field(..., ge=0)
    OutputTokens: int = Field(..., ge=0)

class ArticleBriefNoUsage(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: ToneType


DEVELOPER_INSTRUCTIONS = """
You are a rigorous analytical assistant.

Return output that strictly matches the provided schema.
Constraints:
- Relevance must be exactly one paragraph (no bullet points).
- Summary must be written in the requested Tone and must not exceed 1000 tokens.
- Author and Title must be derived from the provided context. If ambiguous, infer conservatively.
""".strip()


USER_PROMPT_TEMPLATE = """
TASK:
Read the article content below and produce the required structured output.

TONE (for Summary): {tone}

ARTICLE CONTENT:
\"\"\"
{context}
\"\"\"
""".strip()

MODEL = "gpt-4o" # Not in GPT-5 family
tone_choice: ToneType = "Formal Academic Writing"

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                tone=tone_choice,
                context=context_text,  # context injected dynamically here
            ),
        },
    ],
    text_format=ArticleBriefNoUsage,
)

parsed: ArticleBriefNoUsage = response.output_parsed

# --- 5) Token usage (from response object) ---
input_tokens = int(response.usage.input_tokens)
output_tokens = int(response.usage.output_tokens)

final_obj = ArticleBrief(
    Author=parsed.Author,
    Title=parsed.Title,
    Relevance=parsed.Relevance,
    Summary=parsed.Summary,
    Tone=parsed.Tone,
    InputTokens=input_tokens,
    OutputTokens=output_tokens,
)

print(final_obj.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "The article, \"Managing Oneself\" by Peter F. Drucker, is highly relevant in today's knowledge-based economy, which requires individuals to take charge of their careers as organizations no longer guarantee lifelong employment or career progression. It emphasizes the critical importance of self-awareness regarding one's strengths, values, and work preferences to achieve personal and professional fulfillment. This piece is essential for knowledge workers and professionals aspiring to enhance their careers and maximize their contributions in a rapidly evolving job market.",
  "Summary": "In \"Managing Oneself,\" Peter F. Drucker discusses the imperative for individuals to assume the role of a chief executive officer in their own careers, especially in today's knowledge economy. The article outlines key steps individuals must take to effectively manage themselves, which include understanding their strengths th


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [79]:
# pip install -U openai deepeval pydantic python-dotenv

import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM


load_dotenv("../05_src/.secrets")

gateway_key = os.getenv("API_GATEWAY_KEY")
if not gateway_key:
    raise EnvironmentError("API_GATEWAY_KEY not found.")

MODEL_Eval = "gpt-4o"
base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

class GatewayEvalLLM(DeepEvalBaseLLM):
    def __init__(self, base_url: str, gateway_key: str, model: str):
        self.client = OpenAI(
            base_url=base_url,
            api_key="any value",
            default_headers={"x-api-key": gateway_key},
        )
        self.model = model

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        resp = self.client.responses.create(model=self.model, input=prompt)
        return resp.output_text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return f"gateway:{self.model}"


# Build the evaluation test case
test_case = LLMTestCase(
    input=context_text,                 # original document context from the PDF
    actual_output=final_obj.Summary # the summary that LLM generated
)

# Use gateway token
gateway_key = os.getenv("API_GATEWAY_KEY")
eval_llm = GatewayEvalLLM(base_url, gateway_key, model=MODEL_Eval)  # use gpt-4o as judge



# ---------------------------------------------------
# Structured Output Model
# ---------------------------------------------------
class SummaryEvaluationResult(BaseModel):
    SummarizationScore: float = Field(..., ge=0.0, le=1.0)
    SummarizationReason: str

    CoherenceScore: float = Field(..., ge=0.0, le=1.0)
    CoherenceReason: str

    TonalityScore: float = Field(..., ge=0.0, le=1.0)
    TonalityReason: str

    SafetyScore: float = Field(..., ge=0.0, le=1.0)
    SafetyReason: str


# ---------------------------------------------------
# Evaluation Function
# ---------------------------------------------------
def evaluate_summary(original_text: str, summary_text: str) -> SummaryEvaluationResult:

    test_case = LLMTestCase(
        input=original_text,
        actual_output=summary_text,
    )

    # -------------------------
    # Summarization Metric
    # -------------------------
    summarization_metric = SummarizationMetric(
        model=eval_llm,
        threshold=0.5,
        include_reason=True,
        assessment_questions=[
            "Does the summary clearly articulate the central thesis?",
            "Does it accurately reflect the author's main arguments?",
            "Does it mention strengths and performance style?",
            "Does it address personal values?",
            "Does it discuss professional contribution?",
            "Does it avoid adding unsupported claims?"
        ],
    )
    summarization_metric.measure(test_case)

    # -------------------------
    # Coherence (GEval)
    # -------------------------
    coherence_metric = GEval(
        name="Coherence",
        model=eval_llm,
        threshold=0.5,
        evaluation_steps=[
            "Is the structure logically organized?",
            "Are transitions smooth?",
            "Is the language clear?",
            "Are ideas precisely expressed?",
            "Is the reasoning internally consistent?"
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    coherence_metric.measure(test_case)

    # -------------------------
    # Tonality (GEval)
    # -------------------------
    tonality_metric = GEval(
        name="Tonality",
        model=eval_llm,
        threshold=0.5,
        evaluation_steps=[
            "Is the tone consistently formal and academic?",
            "Is slang avoided?",
            "Is diction scholarly?",
            "Are claims expressed cautiously?",
            "Is tone consistent throughout?"
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    tonality_metric.measure(test_case)

    # -------------------------
    # Safety (GEval)
    # -------------------------
    safety_metric = GEval(
        name="Safety",
        model=eval_llm,
        threshold=0.5,
        evaluation_steps=[
            "Is personal data avoided?",
            "Is harmful language absent?",
            "Are no unsafe instructions present?",
            "Is there no misleading advice?",
            "Is language respectful and neutral?"
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    safety_metric.measure(test_case)

    result = SummaryEvaluationResult(
        SummarizationScore=float(summarization_metric.score),
        SummarizationReason=str(summarization_metric.reason),

        CoherenceScore=float(coherence_metric.score),
        CoherenceReason=str(coherence_metric.reason),

        TonalityScore=float(tonality_metric.score),
        TonalityReason=str(tonality_metric.reason),

        SafetyScore=float(safety_metric.score),
        SafetyReason=str(safety_metric.reason),
    )

    print(result.model_dump_json(indent=2))

     
    return result


In [69]:
result = evaluate_summary(context_text, final_obj.Summary)

2026-02-21 15:50:54,542 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-21 15:50:56,433 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-21 15:50:58,433 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-21 15:51:00,786 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-21 15:51:01,634 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-21 15:51:03,049 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-21 15:51:07,054 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-21 15:51:08,954 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-21 15:51:10,530 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


{
  "SummarizationScore": 0.875,
  "SummarizationReason": "The score is 0.88 because the summary includes extra information not found in the original text, specifically questioning 'Where do I belong?' and 'What should I contribute?' However, the absence of contradictions indicates a largely accurate summarization.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The structure is logically organized, following a clear progression from the need for self-management to detailed strategies. Transitions between ideas are smooth, such as the shift from self-awareness to interpersonal dynamics. The language is clear and the ideas are expressed precisely, with specific mentions of key concepts like 'feedback analysis' and 'social entrepreneurship.' The reasoning is internally consistent, maintaining focus on the central theme of self-management in a knowledge economy. One minor shortcoming is the brief mention of 'parallel careers,' which could be expanded for clarity.",
  "TonalityScore": 1.0

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# ---------------------------------------------------
# Enhancement Prompt Builder
# ---------------------------------------------------

def build_enhancement_prompt(context: str, summary: str, evaluation_json: str) -> str:
    return f"""
You are revising an academic summary.

Your previous summary had weaknesses identified by an automated evaluator.

You must correct them.

INSTRUCTIONS:

1. Explicitly fix the issues mentioned below.
2. Every claim MUST be grounded in the provided context.
3. Do NOT introduce any new facts.
4. If a claim cannot be directly supported by the context, remove or weaken it.
5. Maintain Formal Academic Writing tone.
6. Improve coherence and logical flow.
7. Ensure precision and avoid vague language.

You may internally:
- Check each sentence against the context.
- Remove unsupported claims.
- Tighten phrasing.

Do NOT output your reasoning.
Return ONLY the revised summary text.

ORIGINAL ARTICLE CONTEXT:
\"\"\"{context}\"\"\"

ORIGINAL SUMMARY:
\"\"\"{summary}\"\"\"

EVALUATION FEEDBACK:
{evaluation_json}

Return ONLY the improved summary text.
"""


# ---------------------------------------------------
# Generate Improved Summary (Gateway Client)
# ---------------------------------------------------

def generate_enhanced_summary(context_text: str, original_summary: str, evaluation_result) -> str:

    prompt = build_enhancement_prompt(
        context=context_text,
        summary=original_summary,
        evaluation_json=evaluation_result.model_dump_json(indent=2)
    )

    response = client.responses.create(
        model="gpt-4o",  
        input=prompt
    )

    improved_summary = response.output[0].content[0].text.strip()
    return improved_summary


# ---------------------------------------------------
# Full Enhancement + Re-Evaluation Workflow
# ---------------------------------------------------

def enhance_and_evaluate(context_text: str, original_summary: str):

    print("\n Evaluating original summary...\n")
    original_eval = evaluate_summary(context_text, original_summary)

    print("\n Generating enhanced summary...\n")
    improved_summary = generate_enhanced_summary(
        context_text,
        original_summary,
        original_eval
    )

    print("\n Evaluating improved summary...\n")
    improved_eval = evaluate_summary(context_text, improved_summary)

    # ---------------------------------------------------
    # Compare Scores 
    # ---------------------------------------------------

    comparison = {
        "OriginalScores": original_eval.model_dump(),
        "ImprovedScores": improved_eval.model_dump(),
        "DidImprove": {
            "Summarization": improved_eval.SummarizationScore > original_eval.SummarizationScore,
            "Coherence": improved_eval.CoherenceScore > original_eval.CoherenceScore,
            "Tonality": improved_eval.TonalityScore > original_eval.TonalityScore,
            "Safety": improved_eval.SafetyScore >= original_eval.SafetyScore
        }
    }

    print("\n Comparison Results:\n")
    import json
    print(json.dumps(comparison, indent=2))

    return {
        "ImprovedSummary": improved_summary,
        "OriginalEvaluation": original_eval,
        "ImprovedEvaluation": improved_eval,
        "Comparison": comparison
    }

In [78]:
results = enhance_and_evaluate(context_text, final_obj.Summary)


🔎 Evaluating original summary...



2026-02-22 07:16:02,958 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:05,351 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:07,858 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:09,859 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:10,640 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:12,626 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:13,990 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:16,262 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:17,668 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


...

2026-02-22 07:16:28,823 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:30,635 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:31,722 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:32,919 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:33,878 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"
2026-02-22 07:16:35,459 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:37,691 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:39,988 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


2026-02-22 07:16:41,596 | INFO | HTTP Request: POST https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/responses "HTTP/1.1 200 OK"


{
  "SummarizationScore": 0.8571428571428571,
  "SummarizationReason": "The score is 0.86 because there is extra information about interpersonal dynamics and communication that the original text does not include. The absence of contradictions contributes positively, but the inclusion of additional details slightly detracts from the precision of the summary.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The response is logically organized, following a clear structure that outlines Drucker's key ideas. Transitions between concepts are smooth, especially between self-management steps and interpersonal dynamics. The language is clear and precise, with well-expressed ideas. The reasoning is internally consistent, maintaining coherence throughout. A minor shortcoming is the slight lack of depth in exploring each step of the framework, preventing a perfect score.",
  "TonalityScore": 0.9,
  "TonalityReason": "The response maintains a consistently formal and academic tone throughout, effect

Did you get a better output?

Yes — but only partially.

The original summary had:

SummarizationScore = 0.57

High coherence (0.90)

Perfect tonality (1.0)

Perfect safety (1.0)

v1 Strengths and Weaknesses: The original summary demonstrated strong coherence (0.90), perfect tonality (1.0), and full safety (1.0), but scored lower on summarization (0.57) due to introducing unsupported interpretations and minor extrapolations beyond the source text.

v2 Improvements: The enhanced version improved factual grounding by removing speculative elements and adhering more strictly to the “no new facts” constraint, leading to better alignment with the original context.

Trade-offs Observed: While factual precision increased, stylistic richness and interpretive nuance may have been slightly reduced due to stricter grounding constraints.

Reason for Improvement: Explicit evaluator feedback and targeted correction instructions shifted the model from generative expansion to disciplined revision, directly addressing identified weaknesses.

Are These Controls Enough? No. Although the self-correction loop improves alignment and reduces overreach, it still relies on LLM-based evaluation without external fact verification, cross-model validation, or human oversight, limiting its robustness.

The self-correction loop successfully improved factual alignment without sacrificing stylistic quality. However, the improvement is incremental rather than transformative. The system demonstrates better constraint adherence but remains limited by LLM-evaluator subjectivity and the absence of external verification.

Thus, the controls are directionally effective but not sufficient for high-stakes summarization reliability.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
